In [1]:
!pip install pandas numpy google-cloud-storage nltk



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import re
from google.cloud import storage
import io

client = storage.Client()
bucket_name = "data_cleaing"
file_name = "discharge.csv"

In [2]:
bucket = client.bucket(bucket_name)
blob = bucket.blob(file_name)
data = blob.download_as_text()
df = pd.read_csv(io.StringIO(data))

In [3]:
df.head()

,note_id,subject_id,hadm_id,note_type,note_seq,charttime,storetime,text
0,10000032-DS-21,10000032,22595853,DS,21,2180-05-07 00:00:00,2180-05-09 15:26:00,\nName: ___ Unit No: _...
1,10000032-DS-22,10000032,22841357,DS,22,2180-06-27 00:00:00,2180-07-01 10:15:00,\nName: ___ Unit No: _...
2,10000032-DS-23,10000032,29079034,DS,23,2180-07-25 00:00:00,2180-07-25 21:42:00,\nName: ___ Unit No: _...
3,10000032-DS-24,10000032,25742920,DS,24,2180-08-07 00:00:00,2180-08-10 05:43:00,\nName: ___ Unit No: _...
4,10000084-DS-17,10000084,23052089,DS,17,2160-11-25 00:00:00,2160-11-25 15:09:00,\nName: ___ Unit No: __...


In [4]:
df.columns

Index(['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq',
       'charttime', 'storetime', 'text'],
      dtype='object')

In [5]:
df.shape

(331793, 8)

In [6]:
df.fillna("N/A", inplace=True)
df.replace(r'\s*___\s*', " ** ", regex=True, inplace=True)


In [7]:
pd.set_option('display.max_colwidth', None)

In [8]:
df.head()

note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   
3  10000032-DS-24    10000032  25742920        DS        24   
4  10000084-DS-17    10000084  23052089        DS        17   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   
3  2180-08-07 00:00:00  2180-08-10 05:43:00   
4  2160-11-25 00:00:00  2160-11-25 15:09:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [9]:
df.isnull().sum()

note_id       0
subject_id    0
hadm_id       0
note_type     0
note_seq      0
charttime     0
storetime     0
text          0
dtype: int64

In [10]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9.,:;%()/+\-*']+", " ", text)
    text = re.sub(r" {2,}|\t", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean_text)

In [11]:
df.head()

note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   
3  10000032-DS-24    10000032  25742920        DS        24   
4  10000084-DS-17    10000084  23052089        DS        17   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   
3  2180-08-07 00:00:00  2180-08-10 05:43:00   
4  2160-11-25 00:00:00  2160-11-25 15:09:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [12]:
medical_abbreviations = {
    # General Medical Abbreviations
    "Pt ": "Patient",
    "Hx ": "History",
    "Dx ": "Diagnosis",
    "Tx ": "Treatment",
    "Rx ": "Prescription",
    "Sx ": "Symptoms",
    "Px ": "Prognosis",
    "CC": "Chief Complaint",
    "H&P": "History and Physical",
    "ROS": "Review of Systems",
    "H/O": "History Of",
    " h/o ": "History Of",
    "NKDA": "No Known Drug Allergies",
    "FMH": "Family Medical History",
    "SHx": "Social History",
    "PMH": "Past Medical History",

    # Vital Signs & Measurements
    "BP": "Blood Pressure",
    "HR": "Heart Rate",
    "RR": "Respiratory Rate",
    "O2 Sat": "Oxygen Saturation",
    "SpO2": "Oxygen Saturation",
    " T ": "Temperature",
    "MAP": "Mean Arterial Pressure",
    "RRR": "Regular Rate and Rhythm",
    "EtCO2": "End-Tidal Carbon Dioxide",
    "FiO2": "Fraction of Inspired Oxygen",
    "GCS": "Glasgow Coma Scale",

    # Symptoms & Conditions
    "SOB": "Shortness of Breath",
    "CP": "Chest Pain",
    "DOE": "Dyspnea on Exertion",
    "PND": "Paroxysmal Nocturnal Dyspnea",
    "AMS": "Altered Mental Status",
    "LOC": "Loss of Consciousness",
    "N/V": "Nausea and Vomiting",
    "F/C": "Fever and Chills",
    "BRBPR": "Bright Red Blood Per Rectum",
    " Hemoptysis ": "Coughing up Blood",

    # Lab Tests & Imaging
    "WBC": "White Blood Cell Count",
    "RBC": "Red Blood Cell Count",
    "Hgb": "Hemoglobin",
    "HCT": "Hematocrit",
    "PLT": "Platelet Count",
    "BUN": "Blood Urea Nitrogen",
    "Cr ": "Creatinine",
    "Na ": "Sodium",
    " K ": "Potassium",
    " Cl ": "Chloride",
    " HCO3 ": "Bicarbonate",
    "Ca ": "Calcium",
    "Mg ": "Magnesium",
    "Phos ": "Phosphate",
    "INR": "International Normalized Ratio",
    "PTT": "Partial Thromboplastin Time",
    "LFTs ": "Liver Function Tests",
    "AST": "Aspartate Aminotransferase",  # Newly added
    "ALT": "Alanine Aminotransferase",
    " Alk Phos ": "Alkaline Phosphatase",
    "TBili ": "Total Bilirubin",
    "DBili ": "Direct Bilirubin",
    "Lipase ": "Lipase Enzyme Test",
    "ABG": "Arterial Blood Gas",
    "BMP": "Basic Metabolic Panel",
    "CMP": "Comprehensive Metabolic Panel",
    "UA": "Urinalysis",
    "TSH": "Thyroid-Stimulating Hormone",
    "BNP": "B-Type Natriuretic Peptide",
    "ESR": "Erythrocyte Sedimentation Rate",
    "CRP": "C-Reactive Protein",
    "A1C": "Hemoglobin A1C",

    # Imaging & Procedures
    "CXR": "Chest X-Ray",
    "CT": "Computed Tomography",
    "MRI": "Magnetic Resonance Imaging",
    "US": "Ultrasound",
    "EKG": "Electrocardiogram",
    "Echo ": "Echocardiogram",
    "EGD": "Esophagogastroduodenoscopy",

    # Medications & Prescription Terms
    "Lasix ": "Furosemide",
    "Aldactone ": "Spironolactone",
    "ASA": "Aspirin",
    "APAP": "Acetaminophen",
    "NTG": "Nitroglycerin",
    "Heparin ": "Unfractionated Heparin",
    "Lovenox ": "Enoxaparin",
    "Plavix ": "Clopidogrel",
    "Metop ": "Metoprolol",
    "HCTZ": "Hydrochlorothiazide",
    " q_h ": "Every _ Hours",
    "AC": "Before Meals",
    "PC": "After Meals",
    "HS": "At Bedtime",
    "PR": "Per Rectum",
    "SL": "Sublingual",
    "OD": "Right Eye",
    "OS": "Left Eye",
    "OU": "Both Eyes",

    # Chronic Diseases
    "HTN": "Hypertension",
    "DM": "Diabetes Mellitus",
    "HLD": "Hyperlipidemia",
    "CKD": "Chronic Kidney Disease",
    "ESRD": "End-Stage Renal Disease",
    "CHF": "Congestive Heart Failure",
    "COPD": "Chronic Obstructive Pulmonary Disease",
    "CAD": "Coronary Artery Disease",
    "AFib ": "Atrial Fibrillation",
    "DVT": "Deep Vein Thrombosis",
    "PE": "Pulmonary Embolism",

    # Cardiovascular & Respiratory
    "JVD": "Jugular Venous Distension",
    "NSR": "Normal Sinus Rhythm",
    "STEMI": "ST-Elevation Myocardial Infarction",
    "NSTEMI": "Non-ST Elevation Myocardial Infarction",
    "SVT": "Supraventricular Tachycardia",
    "Vfib": "Ventricular Fibrillation",
    "VTach": "Ventricular Tachycardia",

    # Infectious Diseases
    "HIV": "Human Immunodeficiency Virus",
    "AIDS": "Acquired Immunodeficiency Syndrome",
    "TB": "Tuberculosis",
    "MRSA": "Methicillin-Resistant Staphylococcus Aureus",
    "VRE": "Vancomycin-Resistant Enterococcus",
    "HSV": "Herpes Simplex Virus",
    "VZV": "Varicella-Zoster Virus",
    "CMV": "Cytomegalovirus",
    "HBV": "Hepatitis B Virus",
    "HCV": "Hepatitis C Virus",
    "EBV": "Epstein-Barr Virus",
    "ID": "Infectious Disease",
    "ART": "Antiretroviral Therapy",  # Newly added

    # Psychiatric & Neurological
    "BPD": "Bipolar Disorder",
    "MDD": "Major Depressive Disorder",
    "PTSD": "Post-Traumatic Stress Disorder",
    "GAD": "Generalized Anxiety Disorder",
    "SZ": "Seizure",
    "TBI": "Traumatic Brain Injury",
    "TIA": "Transient Ischemic Attack",
    "CVA": "Cerebrovascular Accident",
    "ICP": "Intracranial Pressure",
    "SAH": "Subarachnoid Hemorrhage",
    "SDH": "Subdural Hemorrhage",
    "CTE": "Chronic Traumatic Encephalopathy",

    # Obstetrics & Gynecology
    "G/P": "Gravida/Para",
    "LMP": "Last Menstrual Period",
    "EDC": "Estimated Date of Confinement",
    "C/S": "Cesarean Section",
    "IUD": "Intrauterine Device",
    "OB": "Obstetrics",
    "GYN": "Gynecology",

    # Surgical & Procedural Terms
    "OR": "Operating Room",
    "NPO": "Nothing by Mouth",
    "I&D": "Incision and Drainage",
    "Bx ": "Biopsy",
    "D&C": "Dilation and Curettage",
    "CABG": "Coronary Artery Bypass Graft",
    "LP": "Lumbar Puncture",
    "ETT": "Endotracheal Tube",

    # Miscellaneous
    " F/U ": "Follow-Up",
    " d/c ": "Discharge",
    " s/p ": "Status Post",
    " c/w ": "Consistent With",
    " w/u ": "Work-Up",
    " r/o ": "Rule Out"
}

In [13]:
def expand_abbreviations(text):
    if isinstance(text, str):
        for key, value in medical_abbreviations.items():
            text = text.replace(key, value)
    return text


df["text"] = df["text"].map(expand_abbreviations)

In [14]:
df.head()

note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   
1  10000032-DS-22    10000032  22841357        DS        22   
2  10000032-DS-23    10000032  29079034        DS        23   
3  10000032-DS-24    10000032  25742920        DS        24   
4  10000084-DS-17    10000084  23052089        DS        17   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   
1  2180-06-27 00:00:00  2180-07-01 10:15:00   
2  2180-07-25 00:00:00  2180-07-25 21:42:00   
3  2180-08-07 00:00:00  2180-08-10 05:43:00   
4  2160-11-25 00:00:00  2160-11-25 15:09:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [18]:
cleaned_file_name = "cleaned_discharge.csv"
df.to_csv(cleaned_file_name, index=False)
cleaned_blob = bucket.blob(cleaned_file_name)
cleaned_blob.upload_from_filename(cleaned_file_name)

print(f"Cleaned data uploaded to gs://{bucket_name}/{cleaned_file_name}")

Cleaned data uploaded to gs://data_cleaing/cleaned_discharge.csv
